# ISAC-LoRa: TGAT + PPO TrainingRun these cells **in order**. Each section is a checkpoint -- don't skipthe smoke test, and don't skip straight to a long PPO run beforeconfirming pretraining actually worked.**What you need before starting:** `graph_observations.csv` and`reliability_examples.csv`, downloaded from your RPi4's`receiver/data/training/` folder (see the project's main instructionsfor the `scp` command).

## 1. Setup

In [ ]:
# Confirm GPU is available (Runtime -> Change runtime type -> T4 GPU, if not already set)import torchprint("CUDA available:", torch.cuda.is_available())print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only -- set Runtime > Change runtime type > T4 GPU")

In [ ]:
!pip install -q stable-baselines3 gymnasium xgboost

In [ ]:
from google.colab import filesprint("Upload graph_observations.csv and reliability_examples.csv now:")uploaded = files.upload()

In [ ]:
# Download the ml_advanced/ code from your project (adjust this to however# you're getting the .py files into Colab -- e.g. upload a zip, or clone# from your own git repo if you've pushed one).from google.colab import filesprint("Upload ml_advanced_code.zip (zip of the receiver/ml_advanced/ folder):")code_upload = files.upload()import zipfilefor fname in code_upload:    with zipfile.ZipFile(fname) as z:        z.extractall(".")print("Code extracted.")

## 2. Smoke test -- ALWAYS run this before real training

In [ ]:
!python3 smoke_test.py

If the smoke test failed, STOP and fix the error before continuing --it means there's a real bug in tgat_model.py that a long training runwould just waste time discovering slowly.

## 3. Stage 1: TGAT supervised pretraining

In [ ]:
!python3 tgat_pretrain.py --csv graph_observations.csv --epochs 50 --out tgat_encoder.pt

Check the val_acc trend above -- it should generally increase thenplateau. If it stays near 50% (random guessing) the whole time, themodel isn't learning anything -- check your data has both success andfailure examples (the script itself checks this and will tell you).

## 4. Stage 2: PPO fine-tuning (start small first!)

In [ ]:
# Start with a SHORT run first to confirm the reward is trending up# before committing 30+ minutes to a long one.!python3 ppo_train.py --csv graph_observations.csv --tgat-checkpoint tgat_encoder.pt \    --timesteps 5000 --out ppo_policy_shortrun.zip

Look at `ep_rew_mean` in the output above. If it's trending upwardacross the run, proceed to the real training run below. If it's flat ordecreasing, something's likely wrong with the reward shaping inppo_env.py -- worth reviewing before spending a long run on it.

In [ ]:
# Real run, once the short test looks reasonable!python3 ppo_train.py --csv graph_observations.csv --tgat-checkpoint tgat_encoder.pt \    --timesteps 50000 --out ppo_policy.zip

## 5. Baselines (for your comparison table)

In [ ]:
print("=== Non-ML baselines ===")!python3 -c "import sys; sys.path.insert(0, '.')from baselines.non_ml_baselines import STRATEGIES, evaluate_strategyimport csvgroups = {}with open('graph_observations.csv', newline='') as f:    for row in csv.DictReader(f):        key = (row['node_id'], row['telegram_id'])        groups.setdefault(key, []).append(row)print(f'{len(groups)} telegrams loaded')for name, fn in STRATEGIES.items():    r = evaluate_strategy(groups, fn)    print(f'{name:25s}: reconstruction_rate={r[\"telegram_reconstruction_rate\"]:.3f} avg_frag_recovery={r[\"avg_fragment_recovery_frac\"]:.3f}')"

In [ ]:
print("=== XGBoost baseline ===")!python3 baselines/xgboost_baseline.py --csv reliability_examples.csv

In [ ]:
print("=== LSTM baseline ===")!python3 baselines/lstm_baseline.py --csv reliability_examples.csv --epochs 30

In [ ]:
print("=== DQN baseline (RL comparison point) ===")!python3 baselines/dqn_baseline.py --csv graph_observations.csv --tgat-checkpoint tgat_encoder.pt --timesteps 50000

## 6. Download trained models back to the RPi4

In [ ]:
from google.colab import filesfiles.download("tgat_encoder.pt")files.download("ppo_policy.zip")print("Copy both files to receiver/ml_advanced/models/ on the RPi4, then restart the receiver.")